In [1]:
import pandas as pd
import numpy as np
import json
import re
from ast import literal_eval
import importlib
from dotenv import load_dotenv
import os
os.chdir('..')


In [2]:
# Load environment variables from .env file
load_dotenv()

True

In [3]:
# Set GEMINI_API_KEY environment variable with your API key
os.environ['GEMINI_API_KEY'] = os.getenv('GEMINI_API_KEY')

In [4]:
import base64
from google import genai
from google.genai import types
from copy import deepcopy

In [5]:
def generate(prompt):
	client = genai.Client(
			api_key=os.environ.get("GEMINI_API_KEY"),
		)

	model = "gemini-2.5-flash"
	contents = [
		types.Content(
			role="user",
			parts=[
				types.Part.from_text(
					text=prompt
				),
			],
		),
	]
	generate_content_config = types.GenerateContentConfig(
		temperature=0.75,
		top_p=0.9,
		top_k=40,
		max_output_tokens=8192,
		thinking_config=types.ThinkingConfig(thinking_budget=-1), # Dynamic thinking = -1, no thinking = 0
		response_mime_type="application/json",
		system_instruction=[
			types.Part.from_text(
				text="""You are an expert in Natural Language Processing. You are also linguist with an expertise in Indonesian and English."""
			),
		],
	)
		
	response = client.models.generate_content(
		model=model, contents=contents, config=generate_content_config
	)
	# print(response.text)
	return response

from typing import List, Dict
import re
def parse_absa_string(text: str) -> List[Dict[str, str]]:
    """
    Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
    Each dictionary contains the tag as the key and the corresponding value.
    For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
    [{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
    {'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

    Args:
        text (str): ABSA string output to be parsed.

    Returns:
        List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

    """
    pattern = r"\[(\w+)\]\s*([^[]+)"
    matches = re.findall(pattern, text)

    result = []
    current_dict = {}

    for tag, content in matches:
        if tag == "SSEP":  # Sentence separator -> Start a new dictionary
            result.append(current_dict)
            current_dict = {}
        else:
            current_dict[tag] = content.strip()

    if current_dict:  # Append the last sentence if it exists
        result.append(current_dict)

    return result

In [6]:
dataset_folder = 'corrected_splitopinion_typocorrected'
lang = 'indo'
split = 'test'
dataset_path = f'hotel_dataset/{lang}/{dataset_folder}/hotel_aste_{split}_augmented{"_noreasoning" if split == "train" else ""}.json'
with open(dataset_path, 'r') as f:
	dataset = json.load(f)


In [7]:
dataset_unique = []
for instance in dataset:
	if instance['element_order'] == 'aos':
		dataset_unique.append(deepcopy(instance))

In [8]:
dataset_single_triplet = []
for instance in dataset_unique:
	triplets = parse_absa_string(instance['target'])
	for triplet in triplets:
		new_instance = deepcopy(instance)
		new_instance['target'] = f"[A] {triplet.get('A', '')} [O] {triplet.get('O', '')} [S] {triplet.get('S', '')}"
		dataset_single_triplet.append(new_instance)

In [9]:
df_dataset = pd.DataFrame(dataset_unique)
df_dataset

,sentence_id,instance_id,task_elements,input,target,element_order
0,3500,17500,aos,pelayanan nya sangat ramah . [A] [O] [S],[A] pelayanan nya [O] sangat ramah [S] positive,aos
1,3501,17505,aos,sayang wifi tidak bagus harus keluar kamar . [...,[A] wifi [O] tidak bagus harus keluar kamar [S...,aos
2,3502,17510,aos,"tulisannya twin bed , tetapi yang ada kamarnya...",[A] kamarnya [O] beda [S] negative,aos
3,3503,17515,aos,"over all baik , hanya sja akan lebih memuaskan...",[A] over all [O] baik [S] positive [SSEP] [A] ...,aos
4,3504,17520,aos,fasilatas sesuia . [A] [O] [S],[A] fasilitas [O] sesuai [S] positive,aos
...,...,...,...,...,...,...
995,4495,22475,aos,"lumayan , harga murah banged . [A] [O] [S]",[A] harga [O] murah banget [S] positive [SSEP]...,aos
996,4496,22480,aos,buat lakilaki dan perempuan yang belum menikah...,[A] null [O] buat laki laki dan perempuan yang...,aos
997,4497,22485,aos,"kamar sangat nyaman dan bersih , sungguh menye...",[A] kamar [O] sangat nyaman [S] positive [SSEP...,aos
998,4498,22490,aos,"kamarnya luas , kasurnya empuk , kamar mandiny...",[A] kamarnya [O] luas [S] positive [SSEP] [A] ...,aos


In [10]:
with open(f'utils_notebooks/prompt_translate/translate-triplet-eng.txt', 'r') as f:
	prompt_template = f.read()
print(prompt_template)

## Task Description
You will be given input-output pairs of Aspect Sentiment Triplet Extraction.
Given an Indonesian text and triplets consist of aspect term, opinion term, and sentiment polarity of the Indonesian text, translate all of them to English.
The order of each triplet is (aspect term, opinion term, sentiment polarity).
Below is the definition of each element in the triplet:
- The aspect term refers to a specific feature, attribute, or aspect of a product or service on which a user can express an opinion. Explicit aspect terms appear explicitly as a substring of the given text. The aspect term might be “NULL” for the implicit aspect.
- The sentiment polarity refers to the degree of positivity, negativity or neutrality expressed in the opinion towards a particular aspect or feature of a product or service, and the available polarities include: “positive”, “negative” and “neutral”. “neutral” means mildly positive or mildly negative.
- The opinion term refers to the sentiment or

### Individual API request

In [36]:
from tqdm import tqdm
from time import sleep
from ast import literal_eval

outputs = {}
outputs_text = {}
for idx, row in tqdm(df_dataset.iterrows(), desc="Translating triplets", total=df_dataset.shape[0]):
	input_text = row['input'].replace('[A] [O] [S]', '').strip()
	triplets = parse_absa_string(row['target'])
	target_text = []
	for triplet in triplets:
		target_text.append(f"({triplet.get('A', 'err_empty')}, {triplet.get('O', 'err_empty')}, {triplet.get('S', 'err_empty')})")
	target_text = '[' + ', '.join(target_text) + ']'
	prompt = prompt_template.replace('{text-input}', input_text).replace('{input-triplets}', target_text)
	while True:
		try:
			output = generate(prompt)
			outputs[idx] = output
			outputs_text[idx] = literal_eval(output.text)
			outputs_text[idx]['text'] = input_text
			outputs_text[idx]['triplets'] = target_text
			break
		except Exception as e:
			print(f"Error: {e}")
			sleep(3.0)  # Wait for 3 seconds before retrying
			continue
	
	# Write to json file after each successful generation
	os.makedirs(f'temp/translation_output/eng/{dataset_folder}', exist_ok=True)
	with open(f'temp/translation_output/eng/{dataset_folder}/{os.path.basename(dataset_path)}', 'w') as f:
		json.dump(outputs_text, f, indent=4, ensure_ascii=False)
	
	if idx == 10:
		break

Translating triplets:   0%|          | 10/2482 [00:59<4:06:51,  5.99s/it]


### Batch API request

In [11]:
inline_requests = []
for idx, row in df_dataset.iterrows():
	input_text = row['input'].replace('[A] [O] [S]', '').strip()
	triplets = parse_absa_string(row['target'])
	target_text = []
	for triplet in triplets:
		target_text.append(f"({triplet.get('A', 'err_empty')}, {triplet.get('O', 'err_empty')}, {triplet.get('S', 'err_empty')})")
	target_text = '[' + ', '.join(target_text) + ']'
	prompt = prompt_template.replace('{text-input}', input_text).replace('{input-triplets}', target_text)
	
	inline_request = {
		"contents": [
			{
				"role": "user",
				"parts": [
					{
						"text": prompt
					}
				]
			}
		],
		"config": {
			"temperature": 0.75,
			"top_p": 0.9,
			"top_k": 40,
			"max_output_tokens": 8192,
			"thinking_config": {
				"thinking_budget": -1
			},
			"response_mime_type": "application/json",
			"system_instruction": [
				{
					"text": "You are an expert in Natural Language Processing. You are also linguist with an expertise in Indonesian and English."
				}
			]
		}
	}
	inline_requests.append(inline_request)

In [12]:
len(inline_requests)

1000

In [13]:
inline_requests[40]

{'contents': [{'role': 'user',
   'parts': [{'text': '## Task Description\nYou will be given input-output pairs of Aspect Sentiment Triplet Extraction.\nGiven an Indonesian text and triplets consist of aspect term, opinion term, and sentiment polarity of the Indonesian text, translate all of them to English.\nThe order of each triplet is (aspect term, opinion term, sentiment polarity).\nBelow is the definition of each element in the triplet:\n- The aspect term refers to a specific feature, attribute, or aspect of a product or service on which a user can express an opinion. Explicit aspect terms appear explicitly as a substring of the given text. The aspect term might be “NULL” for the implicit aspect.\n- The sentiment polarity refers to the degree of positivity, negativity or neutrality expressed in the opinion towards a particular aspect or feature of a product or service, and the available polarities include: “positive”, “negative” and “neutral”. “neutral” means mildly positive or mi

In [14]:
print(inline_requests[41]['contents'][0]['parts'][0]['text'])

## Task Description
You will be given input-output pairs of Aspect Sentiment Triplet Extraction.
Given an Indonesian text and triplets consist of aspect term, opinion term, and sentiment polarity of the Indonesian text, translate all of them to English.
The order of each triplet is (aspect term, opinion term, sentiment polarity).
Below is the definition of each element in the triplet:
- The aspect term refers to a specific feature, attribute, or aspect of a product or service on which a user can express an opinion. Explicit aspect terms appear explicitly as a substring of the given text. The aspect term might be “NULL” for the implicit aspect.
- The sentiment polarity refers to the degree of positivity, negativity or neutrality expressed in the opinion towards a particular aspect or feature of a product or service, and the available polarities include: “positive”, “negative” and “neutral”. “neutral” means mildly positive or mildly negative.
- The opinion term refers to the sentiment or

In [15]:
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

In [16]:
print(f'Creating batch job with {len(inline_requests)} requests.')
inline_batch_job = client.batches.create(
    model="models/gemini-2.5-flash",
    src=inline_requests,
    config={
        'display_name': f"job-translation-{lang}-1",
    },
)

print(f"Created batch job: {inline_batch_job.name}")

Creating batch job with 1000 requests.
Created batch job: batches/5crm0h0t2580p641oyxpdhpku0q5h9o8ahzp


In [17]:
# List all batches
batches = client.batches.list()
for batch in batches:
    print(f"Batch ID: {batch.name}, State: {batch.state.name}")

Batch ID: batches/5crm0h0t2580p641oyxpdhpku0q5h9o8ahzp, State: JOB_STATE_PENDING
Batch ID: batches/sde88za41ogfgscv3eg94ezopyjeirkhiu1x, State: JOB_STATE_SUCCEEDED
Batch ID: batches/evg9i1wyd8gvczvytcxt9omherlhibe0vtum, State: JOB_STATE_SUCCEEDED
Batch ID: batches/xr5ky0oempio6kujvbxemftbm98goz78mw23, State: JOB_STATE_SUCCEEDED


In [18]:
from time import sleep

# Use the name of the job you want to check
# e.g., inline_batch_job.name from the previous step
job_name = "batches/5crm0h0t2580p641oyxpdhpku0q5h9o8ahzp"  # (e.g. 'batches/your-batch-id')
batch_job = client.batches.get(name=job_name)

completed_states = set([
    'JOB_STATE_SUCCEEDED',
    'JOB_STATE_FAILED',
    'JOB_STATE_CANCELLED',
    'JOB_STATE_EXPIRED',
])

print(f"Polling status for job: {job_name}")
batch_job = client.batches.get(name=job_name) # Initial get
while batch_job.state.name not in completed_states:
  print(f"Current state: {batch_job.state.name}")
  sleep(30) # Wait for 30 seconds before polling again
  batch_job = client.batches.get(name=job_name)

print(f"Job finished with state: {batch_job.state.name}")
if batch_job.state.name == 'JOB_STATE_FAILED':
    print(f"Error: {batch_job.error}")

Polling status for job: batches/5crm0h0t2580p641oyxpdhpku0q5h9o8ahzp
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Job finished with state: JOB_STATE_SUCCEEDED


In [19]:
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# Use the name of the job you want to check
# e.g., inline_batch_job.name from the previous step
job_name = "batches/5crm0h0t2580p641oyxpdhpku0q5h9o8ahzp"
batch_job = client.batches.get(name=job_name)

outputs = {}
outputs_text = {}

if batch_job.state.name == 'JOB_STATE_SUCCEEDED':

    # If batch job was created with a file
    if batch_job.dest and batch_job.dest.file_name:
        # Results are in a file
        result_file_name = batch_job.dest.file_name
        print(f"Results are in file: {result_file_name}")

        print("Downloading result file content...")
        file_content = client.files.download(file=result_file_name)
        # Process file_content (bytes) as needed
        print(file_content.decode('utf-8'))

    # If batch job was created with inline request
    # (for embeddings, use batch_job.dest.inlined_embed_content_responses)
    elif batch_job.dest and batch_job.dest.inlined_responses:
        # Results are inline
        print("Results are inline:")
        for i, inline_response in enumerate(batch_job.dest.inlined_responses):
            if inline_response.response:
                # Accessing response, structure may vary.
                try:
                    outputs_text[i] = literal_eval(inline_response.response.text)
                except AttributeError as e:
                    print(f"AttributeError in Response {i+1}: {e}")
                    outputs[i] = inline_response.response  # Fallback
            elif inline_response.error:
                print(f"Error: {inline_response.error}")
    else:
        print("No results found (neither file nor inline).")
else:
    print(f"Job did not succeed. Final state: {batch_job.state.name}")
    if batch_job.error:
        print(f"Error: {batch_job.error}")

Results are inline:


In [20]:
with open(f'temp/translation_output/eng/{dataset_folder}/{os.path.basename(dataset_path)}', 'w') as f:
	json.dump(outputs_text, f, indent=4, ensure_ascii=False)

## Preprocess

In [21]:
with open(f'temp/translation_output/eng/{dataset_folder}/{os.path.basename(dataset_path)}', 'r') as f:
	outputs_text = json.load(f)

In [22]:
def add_space_around_punctuation(text):
    # Except for '-'
    # Ensure space before punctuation
    text = re.sub(r'(\S)([.,!?\(\)\"\';:+/]+)', r'\1 \2', text)
    # Ensure space after punctuation
    text = re.sub(r'([.,!?\(\)\"\';:+/]+)(\S)', r'\1 \2', text)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    # Ensure punctuation sequences like '...' are split into spaced dots
    text = re.sub(r'([.]{2,})', lambda m: ' '.join(m.group(1)), text)
    return text.strip()

In [23]:
mismatch_indexes = []
mismatch_notes = {}
for key, instance in outputs_text.items():
	translated_text = add_space_around_punctuation(instance['translated_text'].lower()).strip()
	mismatched = False
	for triplet in instance['translated_triplets']:
		aspect_term = add_space_around_punctuation(triplet['aspect_term'].lower()).strip()
		opinion_term = add_space_around_punctuation(triplet['opinion_term'].lower()).strip()
		if aspect_term not in translated_text and aspect_term != 'null':
			print(f"Mismatch in instance {key}: aspect_term '{aspect_term}' not found in {translated_text}")
			mismatch_notes[key] = mismatch_notes.get(key, []) + [f"aspect_term '{aspect_term}' not found"]
			mismatched = True
		if opinion_term not in translated_text and opinion_term != 'null':
			print(f"Mismatch in instance {key}: opinion_term '{opinion_term}' not found in {translated_text}")
			mismatch_notes[key] = mismatch_notes.get(key, []) + [f"opinion_term '{opinion_term}' not found"]
			mismatched = True
	if mismatched:
		mismatch_indexes.append(key)
print(f"Total mismatches found: {len(mismatch_indexes)}")

Mismatch in instance 24: opinion_term 'not cold enough in room 515' not found in the ac in room 515 is not cold enough .
Mismatch in instance 96: opinion_term 'not available' not found in there were no sandals . they said there was free wifi , but it turned out they didn ' t give the password , which is strange .
Mismatch in instance 106: opinion_term 'already complained but there was no solution' not found in the hot water in the bathroom was not working . we already complained , but there was no solution . we had to bathe in the evening and morning with cold water .
Mismatch in instance 109: opinion_term 'from the outside , it doesn ' t look like a hotel . more like a boarding house .' not found in the pillows are hard and thick . the curtains cannot be closed tightly . there is no hot water . the faucet is broken . there is no sink . the blanket is small . from the outside , it doesn ' t look like a hotel ; it ' s more like a boarding house . the reception area is messy . if a room 

In [24]:
mismatch_indexes = list(set(mismatch_indexes))
len(mismatch_indexes)

47

In [25]:
input_eng = []
for key, instance in outputs_text.items():
	translated_text = add_space_around_punctuation(instance['translated_text'].lower()).strip()
	input_eng.append(f'{translated_text} [A] [O] [S]')

target_eng = []
for key, instance in outputs_text.items():
	triplet_texts = []
	for triplet in instance['translated_triplets']:
		aspect_term = add_space_around_punctuation(triplet['aspect_term'].lower()).strip()
		opinion_term = add_space_around_punctuation(triplet['opinion_term'].lower()).strip()
		sentiment = triplet['sentiment_polarity'].lower().strip()
		triplet_texts.append(f"[A] {aspect_term} [O] {opinion_term} [S] {sentiment}")
	target_eng.append(' [SSEP] '.join(triplet_texts))
print(len(input_eng), len(target_eng))

1000 1000


In [26]:
df_dataset['input_eng'] = input_eng
df_dataset['target_eng'] = target_eng
df_dataset['aspect_or_opinion_not_in_input'] = [str(idx) in mismatch_indexes for idx in df_dataset.index]
df_dataset['mismatch_notes'] = [mismatch_notes.get(str(idx), []) for idx in df_dataset.index]
df_dataset['mismatch_notes_format'] = df_dataset['mismatch_notes'].apply(lambda x: '\n'.join(x))

In [27]:
df_dataset['target_format'] = df_dataset['target'].apply(lambda x: '\n'.join(x.split(' [SSEP] ')))
df_dataset['target_format_eng'] = df_dataset['target_eng'].apply(lambda x: '\n'.join(x.split(' [SSEP] ')))

In [28]:
df_dataset

,sentence_id,instance_id,task_elements,input,target,element_order,input_eng,target_eng,aspect_or_opinion_not_in_input,mismatch_notes,mismatch_notes_format,target_format,target_format_eng
0,3500,17500,aos,pelayanan nya sangat ramah . [A] [O] [S],[A] pelayanan nya [O] sangat ramah [S] positive,aos,the service is very friendly . [A] [O] [S],[A] service [O] very friendly [S] positive,False,[],,[A] pelayanan nya [O] sangat ramah [S] positive,[A] service [O] very friendly [S] positive
1,3501,17505,aos,sayang wifi tidak bagus harus keluar kamar . [...,[A] wifi [O] tidak bagus harus keluar kamar [S...,aos,"unfortunately , the wi-fi is not good ; you ha...",[A] wi-fi [O] not good ; you have to go out of...,False,[],,[A] wifi [O] tidak bagus harus keluar kamar [S...,[A] wi-fi [O] not good ; you have to go out of...
2,3502,17510,aos,"tulisannya twin bed , tetapi yang ada kamarnya...",[A] kamarnya [O] beda [S] negative,aos,"it said twin bed , but the room provided was d...",[A] room [O] different [S] negative,False,[],,[A] kamarnya [O] beda [S] negative,[A] room [O] different [S] negative
3,3503,17515,aos,"over all baik , hanya sja akan lebih memuaskan...",[A] over all [O] baik [S] positive [SSEP] [A] ...,aos,"overall good , it would just be more satisfyin...",[A] overall [O] good [S] positive [SSEP] [A] h...,False,[],,[A] over all [O] baik [S] positive\n[A] air ho...,[A] overall [O] good [S] positive\n[A] hot wat...
4,3504,17520,aos,fasilatas sesuia . [A] [O] [S],[A] fasilitas [O] sesuai [S] positive,aos,the facilities are appropriate . [A] [O] [S],[A] facilities [O] appropriate [S] positive,False,[],,[A] fasilitas [O] sesuai [S] positive,[A] facilities [O] appropriate [S] positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,4495,22475,aos,"lumayan , harga murah banged . [A] [O] [S]",[A] harga [O] murah banget [S] positive [SSEP]...,aos,"it ' s decent , the price is very cheap . [A] ...",[A] price [O] very cheap [S] positive [SSEP] [...,False,[],,[A] harga [O] murah banget [S] positive\n[A] n...,[A] price [O] very cheap [S] positive\n[A] nul...
996,4496,22480,aos,buat lakilaki dan perempuan yang belum menikah...,[A] null [O] buat laki laki dan perempuan yang...,aos,unmarried men and women are also allowed to sh...,[A] null [O] unmarried men and women are also ...,False,[],,[A] null [O] buat laki laki dan perempuan yang...,[A] null [O] unmarried men and women are also ...
997,4497,22485,aos,"kamar sangat nyaman dan bersih , sungguh menye...",[A] kamar [O] sangat nyaman [S] positive [SSEP...,aos,"the room is very comfortable and clean , truly...",[A] room [O] very comfortable [S] positive [SS...,False,[],,[A] kamar [O] sangat nyaman [S] positive\n[A] ...,[A] room [O] very comfortable [S] positive\n[A...
998,4498,22490,aos,"kamarnya luas , kasurnya empuk , kamar mandiny...",[A] kamarnya [O] luas [S] positive [SSEP] [A] ...,aos,"the room is spacious , the bed is soft , and t...",[A] room [O] spacious [S] positive [SSEP] [A] ...,False,[],,[A] kamarnya [O] luas [S] positive\n[A] kasurn...,[A] room [O] spacious [S] positive\n[A] bed [O...


In [29]:
df_dataset[['sentence_id', 'input', 'input_eng', 'target_format', 'target_format_eng', 'aspect_or_opinion_not_in_input', 'mismatch_notes_format']].rename({'target_format': 'target', 'target_format_eng': 'target_eng', 'mismatch_notes_format': 'mismatch_notes'}).to_csv(f'temp/translation_output/eng/{dataset_folder}/translated_dataset_{lang}_{split}.csv', index=False)